# Customer Lifetime Value (CLV) Analysis

This notebook demonstrates how to calculate and predict Customer Lifetime Value using advanced analytics techniques.

## Table of Contents
1. [Introduction to CLV](#introduction)
2. [Data Preparation](#data-preparation)
3. [RFM-based CLV](#rfm-clv)
4. [Predictive CLV Models](#predictive-clv)
5. [Cohort Analysis](#cohort-analysis)
6. [Business Applications](#business-applications)

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")

# Import our custom modules
import sys
sys.path.append('../src')
from data_processor import DataProcessor
from rfm_analyzer import RFMAnalyzer
from clv_calculator import CLVCalculator

## 1. Introduction to CLV {#introduction}

Customer Lifetime Value (CLV) represents the total revenue a business can expect from a customer throughout their entire relationship. Understanding CLV helps businesses:

- **Optimize marketing spend**: Focus on acquiring high-value customers
- **Improve retention**: Identify customers worth investing in
- **Strategic planning**: Forecast revenue and growth
- **Personalization**: Tailor experiences based on customer value

### CLV Calculation Methods:

1. **Historical CLV**: Based on past purchase behavior
2. **Predictive CLV**: Uses machine learning to forecast future value
3. **Cohort-based CLV**: Analyzes customer groups by acquisition time
4. **RFM-based CLV**: Leverages recency, frequency, and monetary patterns

## 2. Data Preparation {#data-preparation}

Let's start by loading and preparing our customer transaction data.

In [ ]:
# Initialize data processor
processor = DataProcessor()

# Generate sample transaction data with more history for CLV analysis
print("Generating sample transaction data for CLV analysis...")
transaction_data = processor.create_sample_dataset(
    n_customers=1500, 
    n_transactions=8000,
    start_date='2020-01-01',  # Longer history for better CLV insights
    end_date='2023-12-31'
)

print(f"Dataset shape: {transaction_data.shape}")
print(f"Date range: {transaction_data['transaction_date'].min()} to {transaction_data['transaction_date'].max()}")
print(f"Number of unique customers: {transaction_data['customer_id'].nunique()}")
print(f"Total revenue: ${transaction_data['amount'].sum():,.2f}")

transaction_data.head()

In [ ]:
# Clean and process the data
clean_data = processor.clean_transaction_data(transaction_data)

# Create customer-level features
customer_features = processor.create_customer_features(clean_data)

print(f"Customer features shape: {customer_features.shape}")
print(f"Average customer value: ${customer_features['total_amount'].mean():.2f}")
print(f"Median customer value: ${customer_features['total_amount'].median():.2f}")

customer_features.head()

In [ ]:
# Analyze customer value distribution
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['Customer Value Distribution', 'Log Customer Value', 
                   'Transaction Frequency', 'Average Order Value']
)

# Customer value distribution
fig.add_trace(
    go.Histogram(x=customer_features['total_amount'], nbinsx=50, name='Customer Value'),
    row=1, col=1
)

# Log customer value (to handle skewness)
fig.add_trace(
    go.Histogram(x=np.log1p(customer_features['total_amount']), nbinsx=50, name='Log Customer Value'),
    row=1, col=2
)

# Transaction frequency
fig.add_trace(
    go.Histogram(x=customer_features['total_transactions'], nbinsx=30, name='Transaction Count'),
    row=2, col=1
)

# Average order value
fig.add_trace(
    go.Histogram(x=customer_features['avg_transaction_amount'], nbinsx=50, name='Avg Order Value'),
    row=2, col=2
)

fig.update_layout(height=600, title_text="Customer Value Analysis", showlegend=False)
fig.show()

# Summary statistics
print("\nCustomer Value Summary Statistics:")
print(customer_features[['total_amount', 'total_transactions', 'avg_transaction_amount']].describe())

## 3. RFM-based CLV {#rfm-clv}

We'll use RFM analysis as a foundation for CLV calculations.

In [ ]:
# Initialize RFM analyzer
rfm_analyzer = RFMAnalyzer()

# Calculate RFM scores
rfm_scores = rfm_analyzer.calculate_rfm(clean_data)
customer_segments = rfm_analyzer.segment_customers(rfm_scores)

# Merge RFM data with customer features
customer_data = customer_features.merge(rfm_scores, on='customer_id', how='left')
customer_data = customer_data.merge(customer_segments[['customer_id', 'segment']], on='customer_id', how='left')

print(f"Combined dataset shape: {customer_data.shape}")
print("\nRFM Segment Distribution:")
print(customer_data['segment'].value_counts())

customer_data.head()

In [ ]:
# Calculate simple CLV based on RFM
# Simple formula: CLV = (Average Order Value) × (Purchase Frequency) × (Customer Lifespan)

# Estimate customer lifespan based on recency patterns
# Customers with low recency are likely to continue purchasing
customer_data['estimated_lifespan_months'] = np.where(
    customer_data['recency'] <= 30, 24,  # Active customers: 24 months
    np.where(customer_data['recency'] <= 90, 12, 6)  # Less active: 12 or 6 months
)

# Monthly purchase frequency
customer_data['monthly_frequency'] = customer_data['purchase_frequency'] / 12

# Simple CLV calculation
customer_data['simple_clv'] = (
    customer_data['avg_transaction_amount'] * 
    customer_data['monthly_frequency'] * 
    customer_data['estimated_lifespan_months']
)

print(f"Average Simple CLV: ${customer_data['simple_clv'].mean():.2f}")
print(f"Median Simple CLV: ${customer_data['simple_clv'].median():.2f}")
print(f"Total CLV: ${customer_data['simple_clv'].sum():,.2f}")

# CLV by segment
clv_by_segment = customer_data.groupby('segment').agg({
    'simple_clv': ['mean', 'median', 'sum', 'count']
}).round(2)

clv_by_segment.columns = ['Mean CLV', 'Median CLV', 'Total CLV', 'Customer Count']
clv_by_segment = clv_by_segment.sort_values('Mean CLV', ascending=False)

print("\nCLV by Customer Segment:")
print(clv_by_segment)

In [ ]:
# Visualize CLV by segment
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Average CLV by Segment', 'CLV Distribution']
)

# Bar chart of average CLV by segment
fig.add_trace(
    go.Bar(
        x=clv_by_segment.index,
        y=clv_by_segment['Mean CLV'],
        name='Average CLV',
        marker_color='lightblue'
    ),
    row=1, col=1
)

# Box plot of CLV distribution
for segment in customer_data['segment'].unique():
    segment_data = customer_data[customer_data['segment'] == segment]['simple_clv']
    fig.add_trace(
        go.Box(y=segment_data, name=segment, showlegend=False),
        row=1, col=2
    )

fig.update_layout(height=400, title_text="Customer Lifetime Value Analysis")
fig.show()

## 4. Predictive CLV Models {#predictive-clv}

Now let's build machine learning models to predict future customer value.

In [ ]:
# Initialize CLV calculator
clv_calculator = CLVCalculator()

# Prepare data for predictive modeling
# We'll predict future 12-month CLV based on historical behavior

# Split data temporally - use first 3 years to predict last year
split_date = pd.to_datetime('2023-01-01')
train_data = clean_data[clean_data['transaction_date'] < split_date]
future_data = clean_data[clean_data['transaction_date'] >= split_date]

print(f"Training period: {train_data['transaction_date'].min()} to {train_data['transaction_date'].max()}")
print(f"Prediction period: {future_data['transaction_date'].min()} to {future_data['transaction_date'].max()}")
print(f"Training transactions: {len(train_data)}")
print(f"Future transactions: {len(future_data)}")

In [ ]:
# Create features from training period
train_features = processor.create_customer_features(train_data)
train_rfm = rfm_analyzer.calculate_rfm(train_data)

# Calculate actual future CLV (target variable)
future_clv = future_data.groupby('customer_id').agg({
    'amount': 'sum'
}).rename(columns={'amount': 'future_clv'}).reset_index()

# Merge training features with future CLV
modeling_data = train_features.merge(train_rfm, on='customer_id', how='left')
modeling_data = modeling_data.merge(future_clv, on='customer_id', how='left')

# Fill missing future CLV with 0 (customers who didn't purchase)
modeling_data['future_clv'] = modeling_data['future_clv'].fillna(0)

print(f"Modeling dataset shape: {modeling_data.shape}")
print(f"Customers with future purchases: {(modeling_data['future_clv'] > 0).sum()}")
print(f"Customers without future purchases: {(modeling_data['future_clv'] == 0).sum()}")
print(f"Average future CLV: ${modeling_data['future_clv'].mean():.2f}")

modeling_data.head()

In [ ]:
# Prepare features for machine learning
feature_columns = [
    'total_transactions', 'total_amount', 'avg_transaction_amount',
    'days_since_first_purchase', 'purchase_frequency',
    'recency', 'frequency', 'monetary',
    'r_score', 'f_score', 'm_score'
]

X = modeling_data[feature_columns].fillna(0)
y = modeling_data['future_clv']

# Split into train/test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]} customers")
print(f"Test set size: {X_test.shape[0]} customers")
print(f"Feature columns: {len(feature_columns)}")

In [ ]:
# Train multiple regression models
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': xgb.XGBRegressor(n_estimators=100, random_state=42)
}

# Scale features for linear regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for linear regression
    if name == 'Linear Regression':
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    model_results[name] = {
        'model': model,
        'predictions': y_pred,
        'mse': mse,
        'rmse': rmse,
        'mae': mae,
        'r2': r2
    }
    
    print(f"RMSE: ${rmse:.2f}")
    print(f"MAE: ${mae:.2f}")
    print(f"R² Score: {r2:.4f}")

In [ ]:
# Compare model performance
performance_df = pd.DataFrame({
    'Model': list(model_results.keys()),
    'RMSE': [results['rmse'] for results in model_results.values()],
    'MAE': [results['mae'] for results in model_results.values()],
    'R² Score': [results['r2'] for results in model_results.values()]
})

performance_df = performance_df.sort_values('R² Score', ascending=False)
print("Model Performance Comparison:")
print(performance_df.round(4))

# Visualize model comparison
fig = px.bar(
    performance_df,
    x='Model',
    y='R² Score',
    title='CLV Prediction Model Performance (R² Score)',
    color='R² Score',
    color_continuous_scale='viridis'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Analyze predictions vs actual for best model
best_model_name = performance_df.iloc[0]['Model']
best_predictions = model_results[best_model_name]['predictions']

# Create prediction vs actual plot
fig = go.Figure()

# Scatter plot of predictions vs actual
fig.add_trace(go.Scatter(
    x=y_test,
    y=best_predictions,
    mode='markers',
    name='Predictions',
    marker=dict(opacity=0.6)
))

# Perfect prediction line
max_val = max(y_test.max(), best_predictions.max())
fig.add_trace(go.Scatter(
    x=[0, max_val],
    y=[0, max_val],
    mode='lines',
    name='Perfect Prediction',
    line=dict(dash='dash', color='red')
))

fig.update_layout(
    title=f'CLV Predictions vs Actual - {best_model_name}',
    xaxis_title='Actual Future CLV ($)',
    yaxis_title='Predicted Future CLV ($)',
    height=500
)
fig.show()

print(f"\nBest Model: {best_model_name}")
print(f"R² Score: {model_results[best_model_name]['r2']:.4f}")
print(f"RMSE: ${model_results[best_model_name]['rmse']:.2f}")

## 5. Cohort Analysis {#cohort-analysis}

Cohort analysis helps understand customer behavior patterns over time.

In [ ]:
# Create cohort analysis
def create_cohort_analysis(data):
    # Get customer's first purchase date
    customer_cohorts = data.groupby('customer_id')['transaction_date'].min().reset_index()
    customer_cohorts.columns = ['customer_id', 'cohort_month']
    customer_cohorts['cohort_month'] = customer_cohorts['cohort_month'].dt.to_period('M')
    
    # Merge cohort info with transaction data
    data_cohort = data.merge(customer_cohorts, on='customer_id')
    data_cohort['transaction_month'] = data_cohort['transaction_date'].dt.to_period('M')
    
    # Calculate period number (months since first purchase)
    data_cohort['period_number'] = (
        data_cohort['transaction_month'] - data_cohort['cohort_month']
    ).apply(attrgetter('n'))
    
    return data_cohort

from operator import attrgetter

# Create cohort data
cohort_data = create_cohort_analysis(clean_data)

print(f"Cohort analysis data shape: {cohort_data.shape}")
print(f"Cohort months: {cohort_data['cohort_month'].nunique()}")
print(f"Max period: {cohort_data['period_number'].max()} months")

cohort_data.head()

In [ ]:
# Calculate cohort retention rates
cohort_sizes = cohort_data.groupby('cohort_month')['customer_id'].nunique().reset_index()
cohort_sizes.columns = ['cohort_month', 'total_customers']

# Number of customers who made purchases in each period
cohort_table = cohort_data.groupby(['cohort_month', 'period_number'])['customer_id'].nunique().reset_index()
cohort_table = cohort_table.merge(cohort_sizes, on='cohort_month')
cohort_table['retention_rate'] = cohort_table['customer_id'] / cohort_table['total_customers']

# Pivot to create retention matrix
retention_matrix = cohort_table.pivot_table(
    index='cohort_month',
    columns='period_number',
    values='retention_rate',
    fill_value=0
)

print("Cohort Retention Matrix (first 6 months):")
print(retention_matrix.iloc[:, :6].round(3))

In [ ]:
# Visualize cohort retention heatmap
import plotly.figure_factory as ff

# Create heatmap of retention rates
fig = px.imshow(
    retention_matrix.values,
    labels=dict(x="Period (Months)", y="Cohort Month", color="Retention Rate"),
    x=retention_matrix.columns,
    y=[str(idx) for idx in retention_matrix.index],
    title="Customer Retention Cohort Analysis",
    color_continuous_scale='Blues'
)

fig.update_layout(height=500)
fig.show()

# Calculate average retention by period
avg_retention = retention_matrix.mean().round(3)
print("\nAverage Retention Rates by Period:")
for period, retention in avg_retention.head(12).items():
    print(f"Month {period}: {retention:.1%}")

In [ ]:
# Calculate cohort CLV over time
cohort_revenue = cohort_data.groupby(['cohort_month', 'period_number'])['amount'].sum().reset_index()
cohort_revenue = cohort_revenue.merge(cohort_sizes, on='cohort_month')
cohort_revenue['revenue_per_customer'] = cohort_revenue['amount'] / cohort_revenue['total_customers']

# Calculate cumulative CLV by cohort
cohort_revenue_matrix = cohort_revenue.pivot_table(
    index='cohort_month',
    columns='period_number',
    values='revenue_per_customer',
    fill_value=0
)

# Cumulative revenue per customer
cumulative_clv = cohort_revenue_matrix.cumsum(axis=1)

print("Cumulative CLV by Cohort (first 6 months):")
print(cumulative_clv.iloc[:, :6].round(2))

# Average cumulative CLV by period
avg_cumulative_clv = cumulative_clv.mean()
print("\nAverage Cumulative CLV by Period:")
for period, clv in avg_cumulative_clv.head(12).items():
    print(f"Month {period}: ${clv:.2f}")

In [ ]:
# Visualize CLV curve
fig = go.Figure()

# Add trace for average cumulative CLV
fig.add_trace(go.Scatter(
    x=list(range(len(avg_cumulative_clv.head(24)))),
    y=avg_cumulative_clv.head(24).values,
    mode='lines+markers',
    name='Average Cumulative CLV',
    line=dict(width=3)
))

# Add traces for individual cohorts (sample)
for cohort in cumulative_clv.index[:5]:  # Show first 5 cohorts
    cohort_data_line = cumulative_clv.loc[cohort].head(24)
    fig.add_trace(go.Scatter(
        x=list(range(len(cohort_data_line))),
        y=cohort_data_line.values,
        mode='lines',
        name=f'Cohort {cohort}',
        opacity=0.5
    ))

fig.update_layout(
    title='Customer Lifetime Value Curves by Cohort',
    xaxis_title='Months Since First Purchase',
    yaxis_title='Cumulative CLV ($)',
    height=500
)
fig.show()

## 6. Business Applications {#business-applications}

Let's apply our CLV insights to real business scenarios.

In [ ]:
# Use CLV calculator for business insights
clv_predictions = clv_calculator.predict_clv(clean_data)

# Merge with customer data for analysis
customer_insights = customer_data.merge(
    clv_predictions[['customer_id', 'predicted_clv_1_year']], 
    on='customer_id', 
    how='left'
)

print(f"Customer insights shape: {customer_insights.shape}")
print(f"Average predicted 1-year CLV: ${customer_insights['predicted_clv_1_year'].mean():.2f}")

customer_insights.head()

In [ ]:
# Customer value segmentation for business strategy
customer_insights['clv_segment'] = pd.cut(
    customer_insights['predicted_clv_1_year'],
    bins=[0, 100, 500, 1000, float('inf')],
    labels=['Low Value', 'Medium Value', 'High Value', 'VIP']
)

# CLV segment analysis
clv_segment_analysis = customer_insights.groupby('clv_segment').agg({
    'customer_id': 'count',
    'predicted_clv_1_year': ['mean', 'sum'],
    'total_amount': 'mean',
    'recency': 'mean'
}).round(2)

clv_segment_analysis.columns = ['Customer Count', 'Avg Predicted CLV', 'Total Predicted CLV', 'Avg Historical Value', 'Avg Recency']

print("CLV Segment Analysis:")
print(clv_segment_analysis)

# Calculate segment percentages
segment_pct = (customer_insights['clv_segment'].value_counts() / len(customer_insights) * 100).round(1)
print("\nSegment Distribution:")
for segment, pct in segment_pct.items():
    print(f"{segment}: {pct}%")

In [ ]:
# Marketing budget allocation based on CLV
# Assume different marketing costs and success rates for each segment

marketing_strategy = {
    'VIP': {'cost_per_customer': 200, 'success_rate': 0.8, 'retention_lift': 0.3},
    'High Value': {'cost_per_customer': 100, 'success_rate': 0.6, 'retention_lift': 0.2},
    'Medium Value': {'cost_per_customer': 50, 'success_rate': 0.4, 'retention_lift': 0.15},
    'Low Value': {'cost_per_customer': 20, 'success_rate': 0.2, 'retention_lift': 0.1}
}

# Calculate ROI for each segment
roi_analysis = []
for segment in clv_segment_analysis.index:
    if segment in marketing_strategy:
        strategy = marketing_strategy[segment]
        segment_data = clv_segment_analysis.loc[segment]
        
        customers = segment_data['Customer Count']
        avg_clv = segment_data['Avg Predicted CLV']
        
        # Calculate campaign metrics
        total_cost = customers * strategy['cost_per_customer']
        successful_customers = customers * strategy['success_rate']
        additional_value = successful_customers * avg_clv * strategy['retention_lift']
        
        roi = (additional_value - total_cost) / total_cost * 100
        
        roi_analysis.append({
            'Segment': segment,
            'Customers': customers,
            'Campaign Cost': total_cost,
            'Additional Value': additional_value,
            'ROI (%)': roi
        })

roi_df = pd.DataFrame(roi_analysis)
roi_df = roi_df.sort_values('ROI (%)', ascending=False)

print("Marketing Campaign ROI Analysis:")
print(roi_df.round(1))

# Visualize ROI by segment
fig = px.bar(
    roi_df,
    x='Segment',
    y='ROI (%)',
    title='Marketing Campaign ROI by Customer Segment',
    color='ROI (%)',
    color_continuous_scale='RdYlGn'
)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Customer acquisition strategy based on CLV insights
# Identify characteristics of high-CLV customers for lookalike modeling

high_clv_customers = customer_insights[customer_insights['clv_segment'].isin(['High Value', 'VIP'])]
low_clv_customers = customer_insights[customer_insights['clv_segment'] == 'Low Value']

print("High-CLV Customer Characteristics:")
print("=" * 40)

characteristics = ['avg_transaction_amount', 'purchase_frequency', 'total_transactions', 'r_score', 'f_score', 'm_score']

comparison = pd.DataFrame({
    'Metric': characteristics,
    'High CLV (Mean)': [high_clv_customers[col].mean() for col in characteristics],
    'Low CLV (Mean)': [low_clv_customers[col].mean() for col in characteristics]
})

comparison['Ratio'] = comparison['High CLV (Mean)'] / comparison['Low CLV (Mean)']
comparison = comparison.round(2)

print(comparison)

# Key insights for acquisition
print("\nKey Insights for Customer Acquisition:")
print("1. Target customers with higher average order values")
print("2. Focus on customers who show frequent purchase behavior")
print("3. Look for customers with recent engagement patterns")
print(f"4. High-CLV customers spend {comparison.loc[0, 'Ratio']:.1f}x more per transaction")
print(f"5. High-CLV customers purchase {comparison.loc[1, 'Ratio']:.1f}x more frequently")

In [ ]:
# Executive summary and recommendations
total_customers = len(customer_insights)
total_predicted_clv = customer_insights['predicted_clv_1_year'].sum()
avg_predicted_clv = customer_insights['predicted_clv_1_year'].mean()
vip_customers = len(customer_insights[customer_insights['clv_segment'] == 'VIP'])
vip_value = customer_insights[customer_insights['clv_segment'] == 'VIP']['predicted_clv_1_year'].sum()

print("EXECUTIVE SUMMARY - Customer Lifetime Value Analysis")
print("=" * 60)
print(f"\n📊 BUSINESS METRICS:")
print(f"   Total Customers Analyzed: {total_customers:,}")
print(f"   Total Predicted 1-Year CLV: ${total_predicted_clv:,.0f}")
print(f"   Average Customer Value: ${avg_predicted_clv:.0f}")
print(f"   VIP Customers: {vip_customers} ({vip_customers/total_customers*100:.1f}%)")
print(f"   VIP Customer Value: ${vip_value:,.0f} ({vip_value/total_predicted_clv*100:.1f}% of total)")

print(f"\n🎯 KEY FINDINGS:")
print(f"   1. Top {vip_customers/total_customers*100:.1f}% of customers generate {vip_value/total_predicted_clv*100:.1f}% of predicted value")
print(f"   2. Average customer retention decreases to {avg_retention[6]:.1%} after 6 months")
print(f"   3. {best_model_name} model achieves {model_results[best_model_name]['r2']:.3f} R² for CLV prediction")
print(f"   4. VIP segment shows highest marketing ROI: {roi_df.iloc[0]['ROI (%)']:.1f}%")

print(f"\n🚀 STRATEGIC RECOMMENDATIONS:")
print(f"   1. Prioritize VIP customer retention (${vip_value:,.0f} at risk)")
print(f"   2. Implement targeted campaigns for high-CLV segments")
print(f"   3. Focus acquisition on high-value customer profiles")
print(f"   4. Monitor and improve 6-month retention rates")
print(f"   5. Deploy predictive CLV models for real-time decisions")

print(f"\n💰 INVESTMENT PRIORITIES:")
print(f"   1. VIP Retention: ${roi_df[roi_df['Segment']=='VIP']['Campaign Cost'].iloc[0]:,.0f} investment")
print(f"   2. High-Value Growth: ${roi_df[roi_df['Segment']=='High Value']['Campaign Cost'].iloc[0]:,.0f} investment")
print(f"   3. Predictive Analytics: Expand CLV modeling capabilities")
print(f"   4. Customer Success: Dedicated teams for top segments")

## Summary

This CLV analysis provides a comprehensive framework for understanding and predicting customer value:

### ✅ What We Accomplished:

1. **Multi-Method CLV Calculation**
   - Simple RFM-based CLV for quick insights
   - Machine learning models for predictive accuracy
   - Cohort analysis for behavioral patterns

2. **Business Segmentation**
   - Customer value tiers (Low, Medium, High, VIP)
   - Segment-specific characteristics and behaviors
   - Targeted strategy recommendations

3. **ROI-Driven Marketing Strategy**
   - Campaign cost-benefit analysis by segment
   - Investment prioritization framework
   - Performance tracking metrics

4. **Actionable Insights**
   - Customer acquisition profiles
   - Retention strategy optimization
   - Revenue forecasting capabilities

### 🎯 Next Steps:

1. **Implementation**: Deploy CLV models in production systems
2. **Automation**: Set up regular model retraining and scoring
3. **Integration**: Connect CLV insights to marketing platforms
4. **Experimentation**: A/B test retention strategies by segment
5. **Enhancement**: Incorporate additional data sources (web behavior, support interactions)

### 📈 Business Impact:

- **Revenue Optimization**: Focus resources on highest-value customers
- **Cost Efficiency**: Reduce acquisition costs through better targeting
- **Customer Experience**: Personalize interactions based on value
- **Strategic Planning**: Data-driven growth and retention strategies

The CLV framework provides a solid foundation for customer-centric business decisions and sustainable growth strategies.